# Software profesional en Acústica 2025-26 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [9]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Numerical Solution of the Helmholtz Equation using the Finite Element Method

This notebook illustrates the numerical solution of the wave equation for harmonic excitation using the so called [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM). The method aims at an approximate solution by subdividing the area of interest into smaller parts with simpler geometry, linking these parts together and applying methods from the calculus of variations to solve the problem numerically. The FEM is a well established method for the numerical approximation of the solution of partial differential equations (PDEs). The solutions of PDEs are often known analytically only for rather simple geometries. FEM based simulations allow to gain insights into other more complex cases.

## Problem Statement

The inhomogeneous linear [wave equation](https://en.wikipedia.org/wiki/Wave_equation) is given as

\begin{equation*}
\Delta p(\boldsymbol{x}, t) - \frac{1}{c^2} \frac{\partial^2}{\partial t^2} p(\boldsymbol{x}, t) = - f(\boldsymbol{x}, t) ,
\end{equation*}

where $p(\boldsymbol{x}, t)$ denotes the sound pressure at position $\boldsymbol{x}$, $c$ the speed of sound and $q(\boldsymbol{x}, t)$ the inhomogeneity.
For an harmonic excitation $f(\boldsymbol{x}, t) = \Re \{ F(\boldsymbol{x}, \omega) \mathrm{e}^{\mathrm{j} \omega t} \}$ with frequency $\omega = 2 \pi f$ we choose the Ansatz $p(\boldsymbol{x}, t) = \Re \{ P(\boldsymbol{x}, \omega) \mathrm{e}^{\mathrm{j} \omega t} \}$ for the sound pressure.
Introduction of the complex quantities into the wave equation yields

\begin{equation*}
\Delta P(\boldsymbol{x}, \omega) \mathrm{e}^{\mathrm{j} \omega t} + \frac{\omega^2}{c^2} P(\boldsymbol{x}, \omega) \mathrm{e}^{\mathrm{j} \omega t} = - F(\boldsymbol{x}, \omega) \mathrm{e}^{\mathrm{j} \omega t} ,
\end{equation*}

and canceling out the $\mathrm{e}^{\mathrm{j} \omega t}$ terms yields the [Helmholtz equation](https://en.wikipedia.org/wiki/Helmholtz_equation)

\begin{equation*}
\Delta P(\boldsymbol{x}, \omega) + \frac{\omega^2}{c^2} P(\boldsymbol{x}, \omega) = - F(\boldsymbol{x}, \omega) .
\end{equation*}

We aim for a numerical solution of the Helmholtz equation on the domain $\Omega$ with respect to the homogeneous Dirichlet boundary condition

\begin{equation*}
P(\boldsymbol{x}, \omega) = 0 \qquad \text{for } \boldsymbol{x}\in \partial \Omega
\end{equation*}

or the homogeneous Neumann boundary condition

\begin{equation*}
\frac{\partial}{\partial\boldsymbol{n}} P(\boldsymbol{x}, \omega) = 0 \qquad \text{for } \boldsymbol{x}\in \partial \Omega,
\end{equation*}

where $\partial \Omega$ denotes the boundary of $\Omega$.

## Variational Formulation

The FEM is based on expressing the partial differential equation (PDE) to be solved in its [variational](https://en.wikipedia.org/wiki/Calculus_of_variations) or weak form.
The first step towards this formulation is to multiply the Helmholtz equation by the test function $Q(\boldsymbol{x}, \omega)$

\begin{equation*}
\Delta P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)  + \frac{\omega^2}{c^2} P(\boldsymbol{x}, \omega)  Q(\boldsymbol{x}, \omega) = - F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) ,
\end{equation*}

followed by integration over the domain $V$

\begin{equation*}
\int_{\Omega} \Delta P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x}  + \frac{\omega^2}{c^2} \int_{\Omega} P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} = - \int_{\Omega} F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} ,
\end{equation*}

where $\,\mathrm{d}\boldsymbol{x}$ denotes a suitably chosen differential element for integration.
Application of [Green's first identity](https://en.wikipedia.org/wiki/Green%27s_identities) yields

\begin{align*}
{-}&\int_{\Omega} \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x}  + \int_{\partial\Omega} Q(\boldsymbol{x}, \omega) \frac{\partial}{\partial\boldsymbol{n}}  P(\boldsymbol{x}, \omega)\,\mathrm{d}\boldsymbol{s}\\
&+ \frac{\omega^2}{c^2} \int_{\Omega} P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} = - \int_{\Omega} F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} .
\end{align*}

This way the differential order of the first integral is lowered which is advisable for application of the FEM.
The second integral vanishes as

* the variation formulation requires $Q(\boldsymbol{x}, \omega) = 0$ on $\partial \Omega$ where $P(\boldsymbol{x}, \omega)$ is known - here by the pure Dirichlet boundary condition - or
* due to a pure homogeneous Neumann boundary condition $\frac{\partial}{\partial\boldsymbol{n}} P(\boldsymbol{x}, \omega)$ on $\partial \Omega$.

This results in the variational/weak formulation of the Helmholtz equation

\begin{equation*}
{-} \int_{\Omega} \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x}  + \frac{\omega^2}{c^2} \int_{\Omega} P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} = - \int_{\Omega} F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} .
\end{equation*}

It is common to express the integral equation above in terms of the bilinear $a(P, Q)$ and linear $L(Q)$ forms

\begin{equation*}
a(P, Q) = \frac{\omega^2}{c^2} \int_{\Omega} P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} - \int_{\Omega} \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} ,
\end{equation*}

\begin{equation*}
L(Q) = - \int_{\Omega} F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega) \,\mathrm{d}\boldsymbol{x} ,
\end{equation*}

where

\begin{equation*}
a(P, Q) = L(Q) \qquad \forall Q.
\end{equation*}

## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
The implementation uses complex-valued function spaces via `H1(..., complex=True)`, consistent with a time-harmonic formulation even when the data is real-valued.
It is common in the FEM to denote the solution of the problem by $u$ and the test function by $v$.
The definition of the problem in NGSolve is very close to the mathematical formulation of the problem.

For the subsequent examples the solution of the inhomogeneous wave equation for a point source $F(\boldsymbol{x}) = \delta(\boldsymbol{x}-\mathbf{x_s})$ at position $\mathbf{x_s}$ is computed using the FEM.
A function is defined for this purpose, accompanied by a plotting routine for the resulting sound field.

In [10]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw

def FEM_Helmholtz(mesh, frequency, xs, neumann_bc=True, c=343):
    """Numerical solution of the Helmholtz equation using NGSolve.

    Uses complex-valued H1 space so the formulation is fully complex
    even when the problem data are real.

    Parameters
    ----------
    mesh        : ngsolve.Mesh
    frequency   : float, excitation frequency in Hz
    xs          : (float, float), point-source position (x, y)
    neumann_bc  : bool, True → sound-hard (Neumann) walls
    c           : float, speed of sound (m/s)

    Returns
    -------
    u : ngsolve.GridFunction (complex)
    """
    # squared wavenumber
    k2 = (2 * np.pi * frequency / c) ** 2

    # ── function space: CG order 2, complex-valued ─────────────────────────
    if neumann_bc:
        # no Dirichlet boundaries → free Neumann on all walls
        V = H1(mesh, order=2, complex=True)
    else:
        # Dirichlet on all boundaries
        V = H1(mesh, order=2, complex=True, dirichlet=".*") 

    u = V.TrialFunction()
    v = V.TestFunction()

    # ── bilinear form  a(u,v) = (∇u,∇v) - k² (u,v)  ─────────────────────────
    a = BilinearForm(V)
    a += grad(u) * grad(v) * dx - k2 * u * v * dx
    a.Assemble()

    # ── linear form  L(v) = Dirac's delta at x=xs  ─────────────────────────
    xp, yp = xs
    f = LinearForm(V)
    f += v(xp, yp)
    f.Assemble()

    # Build solution GridFunction and apply point-source right-hand side
    sol = GridFunction(V)
    # ── Blocking Dirichlet boundary conditions ──────────────────────────────
    if not(neumann_bc):
        sol.Set(CF(0.0), definedon=mesh.Boundaries(".*"))

    # ── solve ───────────────────────────────────────────────────────────────
    sol.vec.data = a.mat.Inverse(V.FreeDofs()) * f.vec

    return sol

Compute the mesh

In [11]:
from netgen.geom2d import SplineGeometry

def make_rect_mesh(x0, y0, x1, y1, maxh=0.1):
    """Create a rectangular NGSolve mesh from (x0,y0) to (x1,y1)."""
    geo = SplineGeometry()
    p1 = geo.AppendPoint(x0, y0)
    p2 = geo.AppendPoint(x1, y0)
    p3 = geo.AppendPoint(x1, y1)
    p4 = geo.AppendPoint(x0, y1)
    geo.Append(["line", p1, p2], bc="bottom")
    geo.Append(["line", p2, p3], bc="right")
    geo.Append(["line", p3, p4], bc="top")
    geo.Append(["line", p4, p1], bc="left")

    # Generate mesh
    ngmesh = geo.GenerateMesh(maxh=maxh)
    return Mesh(ngmesh)

# define geometry and mesh  (5 m × 4 m room, fine mesh ≈ mshr resolution 200)
mesh = make_rect_mesh(0, 0, 5, 4, maxh=0.1)

# Plot mesh
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Sound Field in a Rectangular Room with Sound-Hard Boundaries

The two-dimensional sound field in a rectangular room whose height is very small compared to the wavelength and with rigid boundaries (Neumann boundary condition) is computed for the frequency $f=120$ Hz and source position $x_s = (1.2,3.2)$ m.

In [12]:
# compute solution  (Neumann → sound-hard walls)
gfu = FEM_Helmholtz(mesh, 120, (1.2, 3.2), neumann_bc=True)

# plot the real part of the pressure field
Draw(gfu.real, mesh, "solution", height="3vh")

# plot the modulus of the pressure field
Draw(sqrt(gfu.real**2 + gfu.imag**2), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu, mesh, "solution", animate_complex=True, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene

### Sound Field in Two Coupled Rectangular Rooms

The two-dimensional sound field in two coupled rectangular rooms with sound-hard boundaries (Neumann boundary condition) is computed for the frequency $f=120$ Hz and source position $x_s = (2,0.5)$ m. First the geometry is defined and plotted, for which the mesh is generated with a low number of elements for ease of illustration. A higher resolution is then used for the simulations.

In [13]:
def make_coupled_rooms_mesh(maxh=0.1):
    """
    Build the L-shaped coupled-rooms geometry used in the original notebook:
      left room  : [0,3]×[0,4]
      corridor   : [3,3.5]×[1.5,2.5]
      right room : [3.5,6]×[0,4]
    """
    geo = SplineGeometry()
    # vertices going counter-clockwise around the outer boundary
    pts = [
        (0,   0),    # 0
        (3,   0),    # 1
        (3,   1.5),  # 2
        (3.5, 1.5),  # 3
        (3.5, 0),    # 4
        (6,   0),    # 5
        (6,   4),    # 6
        (3.5, 4),    # 7
        (3.5, 2.5),  # 8
        (3,   2.5),  # 9
        (3,   4),    # 10
        (0,   4),    # 11
    ]
    pids = [geo.AppendPoint(x, y) for x, y in pts]
    n = len(pids)
    for i in range(n):
        geo.Append(["line", pids[i], pids[(i + 1) % n]], bc="wall")

    # Generate mesh
    ngmesh = geo.GenerateMesh(maxh=maxh)
    return Mesh(ngmesh)

# Low-resolution mesh just for visualisation
mesh = make_coupled_rooms_mesh(maxh=0.2)

# Plot the mesh
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

In [14]:
# High-resolution mesh for the simulation
mesh = make_coupled_rooms_mesh(maxh=0.05)

# Compute the FEM solution
gfu = FEM_Helmholtz(mesh, 120, (2, 0.5), neumann_bc=True)

# plot the the pressure field animated with the phase values
Draw(gfu, mesh, "solution", animate_complex=True, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene

### Exercise
Re-implement the FEM function ``compute_fem`` modifying the type of finite elements: use 2nd, 3rd, 4th and 5th-order piecewise continuous Lagrange polynomials to approximate the Helmholtz solution. Which is the effect of this selection on the order of convergence of the method? Please, illustrate your answer with a plot of the error as a function of the mesh size $h$ for each type of finite element. Use the method of manufactured solutions to compute the error, i.e., select an analytical solution of the Helmholtz equation and compute the corresponding source term $F(\boldsymbol{x}, \omega)$ for this solution. In this case, use the analytical solution 
\begin{equation*}
P(\boldsymbol{x}, \omega) = \sin(k x) \sin(k y) ,
\end{equation*}
where $k = \omega / c$ is the wavenumber. Use this solution to impose the Dirichlet boundary condition on the two-connected room domainand compute the error of the numerical solution for different mesh sizes $h$ and different orders of finite elements.

In [15]:
## YOUR CODE HERE

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).